In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import h5py 
import glob
import matplotlib.pyplot as plt
import glob
import os
from scipy.io import mmread

In [ ]:
base_path = '/home/EOCRC_atlas/'

In [ ]:
# load pelka data 
# load matrix
X = mmread(os.path.join(base_path, "data/Pelka_et_al/matrix.mtx.gz")).T  # transpose! (important)

# read barcodes 
barcodes = pd.read_csv(os.path.join(base_path, "data/Pelka_et_al/matrix.barcodes.tsv"), sep="\t", header=None)
barcodes.columns = ["barcode"]

# read genes/features
genes = pd.read_csv(os.path.join(base_path, "data/Pelka_et_al/matrix.genes.tsv"), sep="\t", header=None)
genes.columns = ["gene_id", "gene_name"]

# create AnnData
adata = sc.AnnData(X)
adata.obs_names = barcodes["barcode"].values
adata.var_names = genes["gene_name"].values

adata.var_names_make_unique()
print(adata)

In [ ]:
# add .obs metadata 
tsne = pd.read_csv(os.path.join(base_path, "data/Pelka_et_al/crc10x_tSNE_cl_global.tsv"), sep="\t")
tsne = tsne.iloc[1:]
tsne.set_index = tsne["NAME"]

tsne_columns = ["X", "Y"]   
metadata_columns = ["ClusterFull", "ClusterMidway", "ClusterTop"]

tsne = tsne.loc[adata.obs_names]
adata.obsm["X_tsne"] = tsne[tsne_columns].values
adata.obs = adata.obs.join(tsne[metadata_columns])
adata.obs

In [ ]:
# convert tsne string to # 
adata.obsm["X_tsne"] = adata.obsm["X_tsne"].astype(float)

In [ ]:
# add clinical metadata 
meta = pd.read_csv(os.path.join(base_path, "data/Pelka_et_al/metatable_v3_fix_v3.tsv"), sep="\t")
meta = meta.iloc[1:]
meta.set_index = meta["NAME"]

meta = meta.loc[adata.obs_names]
adata.obs = adata.obs.join(meta)
adata.obs

In [ ]:
patient_df.index.duplicated().sum()

In [ ]:
# add other clinical metadata 
patient_df = pd.read_excel(os.path.join(base_path, "data/Pelka_et_al/1-s2.0-S0092867421009454-mmc1.xlsx"), sheet_name="A. Cohort overview")
patient_df = patient_df.rename(columns={"PatientBarcode_SpecimenType": "PatientTypeID"})
patient_df = patient_df.set_index("PatientTypeID")
adata.obs = adata.obs.drop(columns=["Age", "MMR-IHC"])
adata.obs = adata.obs.join(patient_df[['Age', 'MMR-IHC']], on="PatientTypeID")
adata.obs

In [ ]:
# plot some metadata on tsne 
sc.pl.tsne(adata, color="ClusterTop")
sc.pl.tsne(adata, color="donor_id")
sc.pl.tsne(adata, color="SpecimenType")
sc.pl.tsne(adata, color="PatientTypeID")
sc.pl.tsne(adata, color="ClusterFull")

In [ ]:
# fix .obs columns 
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = pd.to_numeric(adata.obs[col], errors='ignore')
        print('fixed')

In [ ]:
# save adata 
adata.write_h5ad(os.path.join(base_path, "data/pelka_full.h5ad"))

# generate files for inferCNV for each sample 

In [ ]:
adata = sc.read_h5ad(os.path.join(base_path, "data/pelka_full.h5ad"))

In [ ]:
# collapse annotation to epi and non epi 
adata.obs['InferCNV_Annotation'] = 'Non-epithelial'
adata.obs.loc[adata.obs['ClusterTop'] == 'Epi', 'InferCNV_Annotation'] = 'Epithelial'
np.unique(adata.obs['InferCNV_Annotation'])

In [ ]:
# check for any samples that have only 1 cell from one of the cell types 
sampleids = np.unique(adata.obs['PatientTypeID'])
cellCounts = adata.obs.groupby(['PatientTypeID', 'InferCNV_Annotation']).size()
cellCounts[cellCounts==1]

In [ ]:
#check normalization
print(adata.X[1:30,1:30])

In [ ]:
sampleids = np.unique(adata.obs['PatientTypeID'])
n_genes_keep = adata.n_vars - 205 # removing programs that were stored 
adata.X = adata.X.tocsr()

In [ ]:
for i in sampleids: 
    print(i)
    tmp = adata[adata.obs['PatientTypeID']==i]
    annot = tmp.obs['InferCNV_Annotation']
    counts =  pd.DataFrame(tmp.X.toarray())
    counts.index  = tmp.obs.index.tolist()
    counts.columns = tmp.var.index.tolist()
    counts = counts.iloc[:, :n_genes_keep]
    counts = counts.T
    counts.to_csv(f'/home/EOCRC_atlas/results/PELKA_inferCNV/{i}_counts.tsv', sep='\t', index=True)
    annot.to_csv(f'/home/EOCRC_atlas/results/PELKA_inferCNV/{i}_annotation.tsv', sep='\t', index=True, header=False)

In [ ]:
# make .tsv file to upload to data table in Terra 
df = pd.DataFrame({
    'entity:sample_id': sampleids,
    'additional_args': ['--ref_group_names="Non-epithelial" --analysis_mode="subclusters" --HMM --denoise --cutoff=0.1 --leiden_function="modularity"'] * len(sampleids),
    'annotations_file': [f'gs://fc-ed878322-6cf6-49f5-8875-f9c237b931f2/results/PELKA_inferCNV/inferCNV_inputs/{i}_annotation.tsv' for i in sampleids],
    'gene_order_file': ['gs://fc-ed878322-6cf6-49f5-8875-f9c237b931f2/results/2025-10-01_YOCRC_inferCNV/inferCNV_inputs/Trinity_CTAT_cnv_hg38_gencode_v27.txt'] * len(sampleids),
    'raw_counts_matrix': [f'gs://fc-ed878322-6cf6-49f5-8875-f9c237b931f2/results/PELKA_inferCNV/inferCNV_inputs/{i}_counts.tsv' for i in sampleids],
})

In [ ]:
# save csv 
df.to_csv('/home/EOCRC_atlas/results/2026-02_11_PELKA_inferCNV/data_table.tsv', sep='\t', index=False, header=True)